# 📈 Optimizador de Portafolio Interactivo
## Aplicación de la Teoría de Markowitz con Activos Argentinos e Internacionales

---

**Materia:** Economía Computacional  
**Universidad:** Universidad Argentina de la Empresa (UADE)  
**Instancia:** Challenge — Inteligencia Artificial aplicada a Economía  
**Año:** 2025

---

### Resumen

Este notebook desarrolla, paso a paso, la lógica completa del Optimizador de Portafolio Interactivo. Se implementa la **Teoría Moderna de Portafolios de Harry Markowitz (1952)** con datos reales descargados desde Yahoo Finance, incluyendo activos argentinos (ADRs en NYSE) e internacionales (acciones, ETFs).

### Contenido

1. Introducción y planteamiento del problema  
2. Importación de librerías  
3. Descarga y validación de datos  
4. Análisis exploratorio (EDA)  
5. Cálculo de retornos y métricas individuales  
6. Matriz de correlaciones  
7. Simulación Monte Carlo  
8. Optimización SLSQP — Frontera Eficiente  
9. Comparación de carteras  
10. Análisis de riesgo (VaR, CVaR, Drawdown)  
11. Comparación Argentina vs. Exterior  
12. Visualizaciones finales  
13. Conclusiones

---
> ⚠️ **Uso educativo:** Los resultados de este notebook son exclusivamente académicos y no constituyen asesoramiento financiero. El desempeño pasado no garantiza resultados futuros.

---
## 1. Introducción y Planteamiento del Problema

### 1.1 ¿Por qué optimizar un portafolio?

Un inversor que combina activos argentinos y del exterior enfrenta tres preguntas simultáneas:

1. **¿Cuánto rinde cada activo?** → retorno esperado
2. **¿Cuánto oscila cada activo?** → volatilidad (riesgo)
3. **¿Cuánto capital asignar a cada uno?** → pesos del portafolio

La respuesta intuitiva de muchos inversores es "igual a todos" (portafolio equiponderado). Markowitz demostró en 1952 que esta respuesta es **sub-óptima**: existe una combinación matemáticamente superior que maximiza el retorno por unidad de riesgo.

### 1.2 El aporte de Markowitz

La **Teoría Moderna de Portafolios (TMP)** establece que:

- El riesgo de un portafolio **no es la suma** de los riesgos individuales
- La **correlación** entre activos es clave: activos poco correlacionados se compensan mutuamente
- Existe una **frontera eficiente**: el conjunto de portafolios que dominan a todos los demás

### 1.3 Relevancia para el mercado argentino

El inversor argentino enfrenta desafíos particulares:
- **Riesgo cambiario**: activos en ARS vs USD
- **Riesgo país**: volatilidad macroeconómica elevada
- **Acceso a mercados**: ADRs argentinos en NYSE permiten diversificación en dólares

Este proyecto permite comparar ADRs argentinos (YPF, GGAL, PAM, etc.) con activos internacionales (AAPL, NVDA, SPY, etc.) y encontrar el portafolio óptimo para cada perfil de inversor.

---
## 2. Importación de Librerías

In [ ]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import yfinance as yf
import scipy.optimize as sco
import warnings
from datetime import datetime, timedelta, date
import itertools

warnings.filterwarnings("ignore")

# Configuración visual
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.family'] = 'DejaVu Sans'

sns.set_style("whitegrid")

DIAS_ANIO = 252  # días hábiles bursátiles

print("✅ Librerías importadas correctamente")
print(f"   numpy   {np.__version__}")
print(f"   pandas  {pd.__version__}")
print(f"   yfinance {yf.__version__}")

---
## 3. Descarga y Validación de Datos

### 3.1 Catálogo de activos

Se definen dos grupos de activos:
- **Argentinos**: ADRs listados en NYSE (cotización en USD, datos disponibles en Yahoo Finance)
- **Internacionales**: acciones tech, ETFs y commodities de referencia

In [ ]:
# ============================================================
# CATÁLOGO DE ACTIVOS
# ============================================================

ACTIVOS_ARGENTINOS = {
    "YPF S.A.":           "YPF",
    "Grupo Galicia":       "GGAL",
    "Pampa Energía":       "PAM",
    "Banco Macro":         "BMA",
    "Mercado Libre":       "MELI",
    "Globant":             "GLOB",
    "Central Puerto":      "CEPU",
}

ACTIVOS_INTERNACIONALES = {
    "Apple":               "AAPL",
    "NVIDIA":              "NVDA",
    "S&P 500 ETF":         "SPY",
    "Gold ETF":            "GLD",
    "JPMorgan":            "JPM",
}

# Selección para este análisis
TICKERS_ARG  = list(ACTIVOS_ARGENTINOS.values())
TICKERS_INT  = list(ACTIVOS_INTERNACIONALES.values())
TICKERS_TODOS = TICKERS_ARG + TICKERS_INT

print("🇦🇷 Activos argentinos:", TICKERS_ARG)
print("🌎 Activos internacionales:", TICKERS_INT)
print(f"📦 Total activos a analizar: {len(TICKERS_TODOS)}")

### 3.2 Descarga de precios históricos

In [ ]:
# ============================================================
# PARÁMETROS DEL PERÍODO
# ============================================================

FECHA_INICIO = "2022-01-01"
FECHA_FIN    = datetime.today().strftime("%Y-%m-%d")
TASA_LIBRE_RIESGO = 0.05  # 5% anual (referencia global)

print(f"📅 Período: {FECHA_INICIO} → {FECHA_FIN}")
print(f"📊 Tasa libre de riesgo: {TASA_LIBRE_RIESGO:.1%}")

In [ ]:
# ============================================================
# DESCARGA DE PRECIOS (Yahoo Finance via yfinance)
# ============================================================
# Se usan precios de CIERRE AJUSTADOS:
# - Ajustados por dividendos
# - Ajustados por splits de acciones
# Esto garantiza comparabilidad a lo largo del tiempo.

raw_data = yf.download(
    TICKERS_TODOS,
    start=FECHA_INICIO,
    end=FECHA_FIN,
    auto_adjust=True,   # precios ajustados
    progress=True,
)

# Extraer columna "Close"
precios = raw_data["Close"].copy()

# Eliminar columnas completamente vacías
precios = precios.dropna(axis=1, how="all")

# Eliminar filas con algún NaN
precios_completos = precios.dropna()

print(f"\n✅ Datos descargados: {precios_completos.shape[0]} días × {precios_completos.shape[1]} activos")
print(f"   Período efectivo: {precios_completos.index[0].date()} → {precios_completos.index[-1].date()}")
print(f"\n📋 Activos disponibles: {list(precios_completos.columns)}")

# Activos que fallaron
tickers_ok      = list(precios_completos.columns)
tickers_fallidos = [t for t in TICKERS_TODOS if t not in tickers_ok]
if tickers_fallidos:
    print(f"\n⚠️  No disponibles: {tickers_fallidos}")

In [ ]:
# ============================================================
# PRECIOS INICIALES Y FINALES POR ACTIVO
# ============================================================

tabla_precios = pd.DataFrame({
    "Precio Inicial (USD)": precios_completos.iloc[0].round(2),
    "Precio Final (USD)":   precios_completos.iloc[-1].round(2),
    "Variación ($)":        (precios_completos.iloc[-1] - precios_completos.iloc[0]).round(2),
})
tabla_precios["Variación (%)"] = (
    (tabla_precios["Precio Final (USD)"] / tabla_precios["Precio Inicial (USD)"]) - 1
).map("{:.1%}".format)

print("Precios por activo:")
print(tabla_precios.to_string())

---
## 4. Análisis Exploratorio de Datos (EDA)

### 4.1 Evolución de precios — Base 100

Normalizar a base 100 permite comparar activos con precios absolutos muy distintos
(ej: AAPL cotiza ~$180, YPF ~$18). La base 100 solo muestra el **cambio porcentual**.

In [ ]:
# ============================================================
# PRECIOS NORMALIZADOS BASE 100
# ============================================================

base100 = (precios_completos / precios_completos.iloc[0]) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Argentinos
colores_arg = ['#FF6B35', '#F9A825', '#FF8F00', '#E64A19', '#BF360C', '#FF7043', '#FFB300']
for i, t in enumerate([t for t in TICKERS_ARG if t in base100.columns]):
    axes[0].plot(base100.index, base100[t], label=t,
                 color=colores_arg[i % len(colores_arg)], linewidth=1.8)
axes[0].axhline(100, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
axes[0].set_title("🇦🇷 Activos Argentinos (ADRs) — Base 100", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Valor (Base 100)")
axes[0].legend(fontsize=9)
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.0f}"))

# Internacionales
colores_int = ['#2196F3', '#00BFA5', '#9C27B0', '#1a7a4a', '#0D47A1']
for i, t in enumerate([t for t in TICKERS_INT if t in base100.columns]):
    axes[1].plot(base100.index, base100[t], label=t,
                 color=colores_int[i % len(colores_int)], linewidth=1.8)
axes[1].axhline(100, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
axes[1].set_title("🌎 Activos Internacionales — Base 100", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Valor (Base 100)")
axes[1].legend(fontsize=9)
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.0f}"))

plt.suptitle(f"Evolución de Precios Normalizados — Período: {FECHA_INICIO} a {FECHA_FIN}",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("grafico_base100.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado como grafico_base100.png")

**Interpretación:** La normalización a base 100 revela que los activos argentinos presentan mayor variabilidad relativa que los internacionales, reflejando el contexto macroeconómico local y el riesgo país. Los ADRs con mejor desempeño tienden a ser empresas con negocios dolarizados o exportadores.

---
## 5. Cálculo de Retornos y Métricas Individuales

### 5.1 ¿Por qué retornos logarítmicos?

Se usan **retornos logarítmicos** en lugar de aritméticos por tres razones:

1. **Aditividad temporal**: el retorno log acumulado en N períodos es la suma de los retornos log diarios  
   → $r_{acum} = \sum_{t=1}^{N} r_t$ (log) vs $\prod_{t=1}^{N} (1 + r_t) - 1$ (aritmético)

2. **Simetría**: una suba de 50% y una baja de 50% son simétricas en log-retornos  
   → $\ln(1.5) \approx 0.405$ y $\ln(0.5) \approx -0.693$, mientras que aritméticamente $+50\%$ y $-50\%$ no se compensan

3. **Normalidad**: los log-retornos se aproximan mejor a una distribución normal, supuesto necesario para el modelo de Markowitz

In [ ]:
# ============================================================
# RETORNOS LOGARÍTMICOS DIARIOS
# ============================================================
# Fórmula: r_t = ln(P_t / P_{t-1})

log_returns = np.log(precios_completos / precios_completos.shift(1)).dropna()

print(f"Dimensiones: {log_returns.shape[0]} días × {log_returns.shape[1]} activos")
print(f"\nPrimeras 3 filas:")
print(log_returns.head(3).round(4).to_string())

In [ ]:
# ============================================================
# MÉTRICAS INDIVIDUALES POR ACTIVO
# ============================================================

def var_historico(retornos, nivel=0.95):
    return float(np.percentile(retornos.dropna(), (1 - nivel) * 100))

def cvar_historico(retornos, nivel=0.95):
    var = var_historico(retornos, nivel)
    return float(retornos[retornos <= var].mean())

def max_drawdown(precios_serie):
    rolling_max = precios_serie.cummax()
    return float(((precios_serie - rolling_max) / rolling_max).min())

def sortino_ratio(retornos, rf=0.0):
    ret_anual = retornos.mean() * DIAS_ANIO
    neg = retornos[retornos < 0]
    down_vol = neg.std() * np.sqrt(DIAS_ANIO) if len(neg) > 0 else 0
    return float((ret_anual - rf) / down_vol) if down_vol > 0 else 0.0

# Construir tabla de métricas
filas = []
for ticker in log_returns.columns:
    ret_s  = log_returns[ticker].dropna()
    prec_s = precios_completos[ticker].dropna()

    ret_anual = float(ret_s.mean() * DIAS_ANIO)
    vol_anual = float(ret_s.std() * np.sqrt(DIAS_ANIO))
    sharpe    = (ret_anual - TASA_LIBRE_RIESGO) / vol_anual if vol_anual > 0 else 0
    ret_acum  = float((prec_s.iloc[-1] / prec_s.iloc[0]) - 1)

    filas.append({
        "Ticker":              ticker,
        "Grupo":               "🇦🇷 ARG" if ticker in TICKERS_ARG else "🌎 INT",
        "Precio Inicial":      round(prec_s.iloc[0], 2),
        "Precio Final":        round(prec_s.iloc[-1], 2),
        "Retorno Acumulado":   ret_acum,
        "Retorno Anualizado":  ret_anual,
        "Volatilidad Anual":   vol_anual,
        "Sharpe Ratio":        round(sharpe, 3),
        "Sortino Ratio":       round(sortino_ratio(ret_s, TASA_LIBRE_RIESGO), 3),
        "Max Drawdown":        round(max_drawdown(prec_s), 4),
        "VaR 95% Diario":      round(var_historico(ret_s), 4),
        "CVaR 95% Diario":     round(cvar_historico(ret_s), 4),
        "Mejor Día":           round(float(ret_s.max()), 4),
        "Peor Día":            round(float(ret_s.min()), 4),
    })

metricas_df = pd.DataFrame(filas).set_index("Ticker")

# Formatear para mostrar
fmt = {c: "{:.2%}" for c in ["Retorno Acumulado", "Retorno Anualizado",
       "Volatilidad Anual", "Max Drawdown", "VaR 95% Diario",
       "CVaR 95% Diario", "Mejor Día", "Peor Día"]}
fmt.update({"Sharpe Ratio": "{:.3f}", "Sortino Ratio": "{:.3f}",
            "Precio Inicial": "${:.2f}", "Precio Final": "${:.2f}"})

print("Tabla de Métricas Individuales:")
print(metricas_df.style.format(fmt).to_string())

In [ ]:
# ============================================================
# VISUALIZACIÓN: RETORNO vs RIESGO
# ============================================================

fig, ax = plt.subplots(figsize=(12, 7))

for ticker in metricas_df.index:
    row   = metricas_df.loc[ticker]
    ret   = row["Retorno Anualizado"]
    vol   = row["Volatilidad Anual"]
    shr   = row["Sharpe Ratio"]
    es_arg = ticker in TICKERS_ARG

    color   = '#FF6B35' if es_arg else '#2196F3'
    marker  = '*'       if es_arg else 'o'
    size    = 300       if es_arg else 200

    ax.scatter(vol, ret, s=size, color=color, marker=marker,
               zorder=5, edgecolors='white', linewidths=1.5, alpha=0.9)
    ax.annotate(ticker, (vol, ret),
                textcoords="offset points", xytext=(8, 4),
                fontsize=9, fontweight='bold', color='#1A1A2E')

# Líneas de referencia
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axhline(TASA_LIBRE_RIESGO, color='green', linestyle=':', linewidth=1,
           alpha=0.6, label=f'Tasa libre de riesgo ({TASA_LIBRE_RIESGO:.1%})')

# Leyenda manual
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='*', color='w', markerfacecolor='#FF6B35',
           markersize=14, label='🇦🇷 Activos Argentinos'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2196F3',
           markersize=10, label='🌎 Activos Internacionales'),
]
ax.legend(handles=legend_elements, fontsize=10)

ax.set_xlabel("Volatilidad Anualizada", fontsize=12)
ax.set_ylabel("Retorno Anualizado", fontsize=12)
ax.set_title("Retorno vs. Riesgo por Activo", fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

plt.tight_layout()
plt.savefig("grafico_retorno_riesgo.png", dpi=150, bbox_inches='tight')
plt.show()

**Interpretación:** El gráfico retorno-riesgo muestra que los activos argentinos (★) tienden a ubicarse en zonas de mayor volatilidad que los internacionales (●), lo cual es consistente con el riesgo país. Sin embargo, algunos ADRs ofrecen retornos superiores que compensan parcialmente ese riesgo adicional. La línea verde indica la tasa libre de riesgo: activos por debajo de esa línea no compensan siquiera el costo de oportunidad.

---
## 6. Matriz de Correlaciones

La correlación entre activos es el insumo clave del modelo de Markowitz.

**Interpretación:**
- $\rho = +1$: se mueven exactamente igual → sin diversificación
- $\rho = 0$: movimientos independientes → diversificación total
- $\rho = -1$: se mueven opuesto → cobertura perfecta (riesgo → 0)

In [ ]:
# ============================================================
# MATRIZ DE CORRELACIONES
# ============================================================

corr_matrix = log_returns.corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.zeros_like(corr_matrix, dtype=bool)  # sin máscara (mostrar todo)

sns.heatmap(
    corr_matrix, annot=True, fmt=".2f",
    cmap="RdYlGn", vmin=-1, vmax=1,
    square=True, linewidths=0.5, linecolor='white',
    cbar_kws={"shrink": 0.8, "label": "Correlación"},
    ax=ax,
)
ax.set_title("Matriz de Correlación de Retornos Logarítmicos Diarios",
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("grafico_correlaciones.png", dpi=150, bbox_inches='tight')
plt.show()

# Par de mayor y menor correlación
pares = []
for a, b in itertools.combinations(log_returns.columns, 2):
    pares.append({"Par": f"{a} — {b}", "Correlación": corr_matrix.loc[a, b]})
df_pares = pd.DataFrame(pares).sort_values("Correlación", ascending=False)
print("Par con MAYOR correlación:", df_pares.iloc[0]["Par"],
      f"({df_pares.iloc[0]['Correlación']:.4f})")
print("Par con MENOR correlación:", df_pares.iloc[-1]["Par"],
      f"({df_pares.iloc[-1]['Correlación']:.4f})")

---
## 7. Simulación Monte Carlo

Se generan **10.000 portafolios aleatorios** asignando pesos al azar que suman 1.
Para cada uno se calcula retorno, volatilidad y Sharpe Ratio.

**Objetivo:** explorar visualmente el espacio de combinaciones posibles y
aproximar los portafolios óptimos antes de la optimización exacta.

In [ ]:
# ============================================================
# FUNCIONES DE PORTAFOLIO
# ============================================================

tickers_ok = list(log_returns.columns)
n_activos  = len(tickers_ok)

def port_retorno(pesos):
    """Retorno anualizado del portafolio."""
    return float(np.dot(pesos, log_returns.mean()) * DIAS_ANIO)

def port_volatilidad(pesos):
    """Volatilidad anualizada del portafolio."""
    cov = log_returns.cov() * DIAS_ANIO
    return float(np.sqrt(np.dot(pesos.T, np.dot(cov.values, pesos))))

def port_sharpe(pesos, rf=TASA_LIBRE_RIESGO):
    """Sharpe Ratio del portafolio."""
    vol = port_volatilidad(pesos)
    return float((port_retorno(pesos) - rf) / vol) if vol > 0 else 0.0

print(f"✅ Funciones definidas para {n_activos} activos: {tickers_ok}")

In [ ]:
# ============================================================
# SIMULACIÓN MONTE CARLO (10.000 portafolios)
# ============================================================

np.random.seed(42)
N_SIM = 10_000

p_pesos, p_ret, p_vol, p_sharpe = [], [], [], []

for _ in range(N_SIM):
    pesos = np.random.uniform(0, 1, n_activos)
    pesos /= pesos.sum()  # normalizar para que sumen 1

    p_pesos.append(pesos)
    p_ret.append(port_retorno(pesos))
    p_vol.append(port_volatilidad(pesos))
    p_sharpe.append(port_sharpe(pesos))

# Convertir a arrays
prets    = np.array(p_ret)
pvols    = np.array(p_vol)
psharpes = np.array(p_sharpe)

# DataFrame de resultados
mc_df = pd.DataFrame({
    "retorno":    prets,
    "volatilidad": pvols,
    "sharpe":     psharpes,
})
for i, t in enumerate(tickers_ok):
    mc_df[f"w_{t}"] = [p[i] for p in p_pesos]

# Identificar mejores portafolios
idx_max_sharpe = mc_df["sharpe"].idxmax()
idx_min_vol    = mc_df["volatilidad"].idxmin()

print(f"✅ {N_SIM:,} portafolios simulados")
print(f"\nPortafolio de Máximo Sharpe (simulación):")
print(f"  Retorno:     {mc_df.loc[idx_max_sharpe, 'retorno']:.2%}")
print(f"  Volatilidad: {mc_df.loc[idx_max_sharpe, 'volatilidad']:.2%}")
print(f"  Sharpe:      {mc_df.loc[idx_max_sharpe, 'sharpe']:.4f}")
print(f"\nPortafolio de Mínima Volatilidad (simulación):")
print(f"  Retorno:     {mc_df.loc[idx_min_vol, 'retorno']:.2%}")
print(f"  Volatilidad: {mc_df.loc[idx_min_vol, 'volatilidad']:.2%}")
print(f"  Sharpe:      {mc_df.loc[idx_min_vol, 'sharpe']:.4f}")

In [ ]:
# ============================================================
# GRÁFICO: NUBE DE PORTAFOLIOS MONTE CARLO
# ============================================================

fig, ax = plt.subplots(figsize=(13, 8))

sc = ax.scatter(pvols, prets, c=psharpes, cmap='RdYlGn',
                s=3, alpha=0.5, rasterized=True)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio', pad=0.02)

# Portafolio Máximo Sharpe
ax.scatter(mc_df.loc[idx_max_sharpe, 'volatilidad'],
           mc_df.loc[idx_max_sharpe, 'retorno'],
           c='gold', s=400, marker='*', zorder=10,
           edgecolors='black', linewidths=1,
           label=f"Máx. Sharpe (sim) = {mc_df.loc[idx_max_sharpe,'sharpe']:.2f}")

# Portafolio Mínima Volatilidad
ax.scatter(mc_df.loc[idx_min_vol, 'volatilidad'],
           mc_df.loc[idx_min_vol, 'retorno'],
           c='deepskyblue', s=400, marker='*', zorder=10,
           edgecolors='black', linewidths=1,
           label=f"Mín. Volatilidad (sim) = {mc_df.loc[idx_min_vol,'volatilidad']:.2%}")

# Activos individuales
for t in tickers_ok:
    ret_t = float(log_returns[t].mean() * DIAS_ANIO)
    vol_t = float(log_returns[t].std() * np.sqrt(DIAS_ANIO))
    color_t = '#FF6B35' if t in TICKERS_ARG else '#1A1A2E'
    ax.scatter(vol_t, ret_t, c=color_t, s=80, zorder=8,
               edgecolors='white', linewidths=1)
    ax.annotate(t, (vol_t, ret_t), xytext=(5, 3),
                textcoords="offset points", fontsize=8, fontweight='bold',
                color=color_t)

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.4)
ax.set_xlabel("Volatilidad Anualizada", fontsize=12)
ax.set_ylabel("Retorno Anualizado", fontsize=12)
ax.set_title(f"Nube de Portafolios — Monte Carlo ({N_SIM:,} simulaciones)",
             fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig("grafico_monte_carlo.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Optimización SLSQP — Frontera Eficiente

La simulación Monte Carlo *aproxima* los portafolios óptimos. La optimización **SLSQP**
(*Sequential Least Squares Programming*) los encuentra con precisión matemática.

### Restricciones del problema

$$\min_w \; -\text{Sharpe}(w) = -\frac{E[r_p] - r_f}{\sigma_p}$$

Sujeto a:
- $\sum_{i=1}^{N} w_i = 1$ (los pesos suman 1)
- $w_i \geq 0$ (sin ventas en corto)

In [ ]:
# ============================================================
# OPTIMIZACIÓN — MÁXIMO SHARPE RATIO
# ============================================================

bounds      = tuple((0.0, 1.0) for _ in range(n_activos))
constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
w0          = np.array([1 / n_activos] * n_activos)  # inicio: equiponderado

# Función objetivo: MINIMIZAR el negativo del Sharpe (equiv. a MAXIMIZAR el Sharpe)
def neg_sharpe(w):
    r = port_retorno(w)
    v = port_volatilidad(w)
    return -(r - TASA_LIBRE_RIESGO) / v if v > 0 else 0.0

resultado_ms = sco.minimize(
    neg_sharpe, w0,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"ftol": 1e-12, "maxiter": 1000},
)

if resultado_ms.success:
    w_ms = resultado_ms.x
    print("✅ Máximo Sharpe — Optimización exitosa")
    print(f"   Retorno:     {port_retorno(w_ms):.2%}")
    print(f"   Volatilidad: {port_volatilidad(w_ms):.2%}")
    print(f"   Sharpe:      {port_sharpe(w_ms):.4f}")
    print("\n   Pesos por activo:")
    for t, w in sorted(zip(tickers_ok, w_ms), key=lambda x: -x[1]):
        barra = '█' * int(w * 30) + '░' * (30 - int(w * 30))
        print(f"   {t:8s} {barra} {w:.2%}")
else:
    print("❌ Optimización no convergió:", resultado_ms.message)
    w_ms = w0

In [ ]:
# ============================================================
# OPTIMIZACIÓN — MÍNIMA VARIANZA
# ============================================================

resultado_mv = sco.minimize(
    port_volatilidad, w0,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"ftol": 1e-12, "maxiter": 1000},
)

if resultado_mv.success:
    w_mv = resultado_mv.x
    print("✅ Mínima Varianza — Optimización exitosa")
    print(f"   Retorno:     {port_retorno(w_mv):.2%}")
    print(f"   Volatilidad: {port_volatilidad(w_mv):.2%}")
    print(f"   Sharpe:      {port_sharpe(w_mv):.4f}")
    print("\n   Pesos por activo:")
    for t, w in sorted(zip(tickers_ok, w_mv), key=lambda x: -x[1]):
        barra = '█' * int(w * 30) + '░' * (30 - int(w * 30))
        print(f"   {t:8s} {barra} {w:.2%}")
else:
    print("❌ Optimización no convergió:", resultado_mv.message)
    w_mv = w0

In [ ]:
# ============================================================
# FRONTERA EFICIENTE
# ============================================================
# Para cada nivel de retorno objetivo, minimizamos la volatilidad.
# La curva resultante es la FRONTERA EFICIENTE.

ret_min = log_returns.mean().min() * DIAS_ANIO
ret_max = log_returns.mean().max() * DIAS_ANIO
retornos_objetivo = np.linspace(ret_min, ret_max, 80)

frontera_vols = []
frontera_rets = []

for ret_obj in retornos_objetivo:
    constraints_fe = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1},
        {"type": "eq", "fun": lambda w, r=ret_obj: port_retorno(w) - r},
    ]
    res = sco.minimize(
        port_volatilidad, w0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints_fe,
        options={"ftol": 1e-10, "maxiter": 500},
    )
    if res.success:
        frontera_vols.append(port_volatilidad(res.x))
        frontera_rets.append(ret_obj)

print(f"✅ Frontera eficiente: {len(frontera_rets)} puntos calculados")

In [ ]:
# ============================================================
# GRÁFICO FINAL: FRONTERA EFICIENTE + MONTE CARLO
# ============================================================

fig, ax = plt.subplots(figsize=(14, 9))

# Nube Monte Carlo
sc = ax.scatter(pvols, prets, c=psharpes, cmap='RdYlGn',
                s=3, alpha=0.4, rasterized=True, zorder=2)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio', pad=0.02)

# Frontera eficiente
ax.plot(frontera_vols, frontera_rets, 'b-', linewidth=3,
        label='Frontera Eficiente', zorder=6)

# Capital Market Line (CML)
v_ms_opt = port_volatilidad(w_ms)
r_ms_opt = port_retorno(w_ms)
s_ms_opt = port_sharpe(w_ms)
vol_cml  = np.linspace(0, max(pvols) * 1.1, 100)
ret_cml  = TASA_LIBRE_RIESGO + s_ms_opt * vol_cml
ax.plot(vol_cml, ret_cml, '--', color='gold', linewidth=1.5,
        alpha=0.8, label=f'Capital Market Line (Sharpe={s_ms_opt:.2f})', zorder=5)

# Máximo Sharpe
ax.scatter(v_ms_opt, r_ms_opt, c='gold', s=450, marker='*',
           zorder=10, edgecolors='black', linewidths=1.5,
           label=f"⭐ Máx. Sharpe: {s_ms_opt:.2f}")

# Mínima Varianza
v_mv_opt = port_volatilidad(w_mv)
r_mv_opt = port_retorno(w_mv)
ax.scatter(v_mv_opt, r_mv_opt, c='deepskyblue', s=450, marker='*',
           zorder=10, edgecolors='black', linewidths=1.5,
           label=f"🛡️ Mín. Var.: {v_mv_opt:.1%}")

# Equiponderado
w_eq      = np.array([1 / n_activos] * n_activos)
v_eq      = port_volatilidad(w_eq)
r_eq      = port_retorno(w_eq)
ax.scatter(v_eq, r_eq, c='#9E9E9E', s=300, marker='D',
           zorder=9, edgecolors='black', linewidths=1,
           label=f"◆ Equiponderado")

# Activos individuales
for t in tickers_ok:
    ret_t = float(log_returns[t].mean() * DIAS_ANIO)
    vol_t = float(log_returns[t].std() * np.sqrt(DIAS_ANIO))
    c_t   = '#FF6B35' if t in TICKERS_ARG else '#0A2342'
    ax.scatter(vol_t, ret_t, c=c_t, s=90, zorder=8,
               edgecolors='white', linewidths=1)
    ax.annotate(t, (vol_t, ret_t), xytext=(6, 4),
                textcoords="offset points", fontsize=8.5,
                fontweight='bold', color=c_t)

ax.axhline(TASA_LIBRE_RIESGO, color='green', linestyle=':',
           linewidth=1, alpha=0.5,
           label=f"Tasa libre de riesgo ({TASA_LIBRE_RIESGO:.1%})")
ax.axhline(0, color='gray', linestyle='--', linewidth=0.6, alpha=0.3)

ax.set_xlabel("Volatilidad Anualizada", fontsize=12)
ax.set_ylabel("Retorno Anualizado", fontsize=12)
ax.set_title("Frontera Eficiente de Markowitz — Optimizador de Portafolio",
             fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig("grafico_frontera_eficiente.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado como grafico_frontera_eficiente.png")

---
## 9. Comparación de Carteras

Se comparan tres portafolios:

| Cartera | Criterio | Descripción |
|---------|----------|-------------|
| **Máximo Sharpe** | $\max$ Sharpe | Mejor relación retorno/riesgo |
| **Mínima Varianza** | $\min \sigma$ | Menor riesgo total |
| **Equiponderado** | $w_i = 1/N$ | Benchmark ingenuo |

In [ ]:
# ============================================================
# TABLA COMPARATIVA DE CARTERAS
# ============================================================

carteras = {
    "Máximo Sharpe":   w_ms,
    "Mínima Varianza": w_mv,
    "Equiponderado":   w_eq,
}

filas_cart = []
for nombre, w in carteras.items():
    # Evolución y drawdown
    ret_d    = (log_returns * w).sum(axis=1)
    evol     = (1 + ret_d).cumprod() * 100
    mdd_cart = float(((evol - evol.cummax()) / evol.cummax()).min())

    fila = {
        "Cartera":            nombre,
        "Retorno Anual":      port_retorno(w),
        "Volatilidad Anual":  port_volatilidad(w),
        "Sharpe Ratio":       port_sharpe(w),
        "Max Drawdown":       mdd_cart,
    }
    for t, wi in zip(tickers_ok, w):
        fila[f"w_{t}"] = round(wi, 4)
    filas_cart.append(fila)

tabla_cart = pd.DataFrame(filas_cart).set_index("Cartera")

# Mostrar métricas principales
cols_met = ["Retorno Anual", "Volatilidad Anual", "Sharpe Ratio", "Max Drawdown"]
fmt_cart = {c: "{:.2%}" for c in cols_met}
fmt_cart["Sharpe Ratio"] = "{:.3f}"
print("Métricas comparativas:")
print(tabla_cart[cols_met].style.format(fmt_cart).to_string())

# Pesos
cols_w = [f"w_{t}" for t in tickers_ok]
print("\nPesos por activo:")
print(tabla_cart[cols_w].style.format("{:.2%}").to_string())

In [ ]:
# ============================================================
# GRÁFICO: EVOLUCIÓN DE LOS TRES PORTAFOLIOS
# ============================================================

fig, ax = plt.subplots(figsize=(14, 6))

estilos = {
    "Máximo Sharpe":   {"color": "gold",       "lw": 3,   "ls": "-"},
    "Mínima Varianza": {"color": "deepskyblue", "lw": 2.5, "ls": "--"},
    "Equiponderado":   {"color": "#9E9E9E",     "lw": 2,   "ls": ":"},
}

for nombre, w in carteras.items():
    ret_d = (log_returns * w).sum(axis=1)
    evol  = (1 + ret_d).cumprod() * 100
    est   = estilos[nombre]
    ax.plot(evol.index, evol, label=nombre,
            color=est["color"], linewidth=est["lw"], linestyle=est["ls"])

ax.axhline(100, color='gray', linestyle='--', linewidth=0.8, alpha=0.4)
ax.set_xlabel("Fecha", fontsize=11)
ax.set_ylabel("Valor (Base 100)", fontsize=11)
ax.set_title("Evolución Histórica de los Tres Portafolios (Base 100)", fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig("grafico_evolucion_carteras.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# GRÁFICO: PESOS DE CADA CARTERA (BARRA APILADA)
# ============================================================

colores_pesos = ['#2196F3','#00BFA5','#FF6B35','#9C27B0','#F9A825',
                 '#1a7a4a','#c0392b','#0A2342','#00ACC1','#FF8F00',
                 '#7B1FA2','#388E3C']

fig, ax = plt.subplots(figsize=(10, 6))
bottom = np.zeros(3)
x_labels = list(carteras.keys())

for i, t in enumerate(tickers_ok):
    vals = [carteras[c][i] * 100 for c in carteras]
    ax.bar(x_labels, vals, bottom=bottom,
           label=t, color=colores_pesos[i % len(colores_pesos)])
    for j, v in enumerate(vals):
        if v > 3:
            ax.text(j, bottom[j] + v / 2, f"{v:.0f}%",
                    ha='center', va='center', fontsize=8.5,
                    fontweight='bold', color='white')
    bottom += np.array(vals)

ax.set_ylabel("Peso (%)", fontsize=11)
ax.set_title("Composición de Pesos por Cartera", fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig("grafico_pesos_carteras.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Análisis de Riesgo

### 10.1 Value at Risk (VaR) y CVaR

**VaR al 95%**: pérdida máxima esperada en el 95% de los días  
→ Si VaR = -3%, en el 5% peor de los días la pérdida supera el 3%

**CVaR (Expected Shortfall)**: pérdida *promedio* en los días peores que el VaR  
→ Más conservador que el VaR: captura el riesgo en la cola

In [ ]:
# ============================================================
# VAR Y CVAR POR ACTIVO
# ============================================================

fig, axes = plt.subplots(2, (n_activos + 1) // 2, figsize=(16, 8))
axes = axes.flatten()

for i, ticker in enumerate(tickers_ok):
    ret_t  = log_returns[ticker].dropna()
    var_95 = np.percentile(ret_t, 5)
    cvar_95 = ret_t[ret_t <= var_95].mean()

    axes[i].hist(ret_t * 100, bins=60, color='#2196F3', alpha=0.7,
                 edgecolor='none')
    axes[i].axvline(var_95 * 100, color='red', linestyle='--', linewidth=1.8,
                    label=f'VaR 95%: {var_95:.2%}')
    axes[i].axvline(cvar_95 * 100, color='darkred', linestyle='-', linewidth=1.5,
                    label=f'CVaR 95%: {cvar_95:.2%}')
    axes[i].set_title(ticker, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=7)
    axes[i].set_xlabel("Retorno diario (%)", fontsize=8)

# Ocultar subplots vacíos
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribución de Retornos Diarios con VaR y CVaR al 95%",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("grafico_var_cvar.png", dpi=150, bbox_inches='tight')
plt.show()

### 10.2 Drawdown Histórico

In [ ]:
# ============================================================
# DRAWDOWN POR ACTIVO
# ============================================================

fig, axes = plt.subplots(2, (n_activos + 1) // 2, figsize=(16, 8), sharex=False)
axes = axes.flatten()

for i, ticker in enumerate(tickers_ok):
    prec_t      = precios_completos[ticker]
    rolling_max = prec_t.cummax()
    dd          = (prec_t - rolling_max) / rolling_max * 100
    mdd         = dd.min()

    color = '#FF6B35' if ticker in TICKERS_ARG else '#2196F3'
    axes[i].fill_between(dd.index, dd, 0, alpha=0.4, color=color)
    axes[i].plot(dd.index, dd, color=color, linewidth=1)
    axes[i].axhline(mdd, color='red', linestyle=':', linewidth=1,
                    label=f'Max DD: {mdd:.1f}%')
    axes[i].set_title(ticker, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel("Drawdown (%)", fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Drawdown Histórico por Activo", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("grafico_drawdown.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Comparación Argentina vs. Exterior

In [ ]:
# ============================================================
# MÉTRICAS PROMEDIO POR GRUPO
# ============================================================

def stats_grupo(tickers_g):
    sub  = metricas_df.loc[[t for t in tickers_g if t in metricas_df.index]]
    if sub.empty:
        return {}
    rets = log_returns[[t for t in tickers_g if t in log_returns.columns]]
    corr_vals = rets.corr().values
    mask  = ~np.eye(corr_vals.shape[0], dtype=bool)
    c_prom = float(corr_vals[mask].mean()) if mask.any() else float("nan")
    return {
        "Retorno anual prom.":      sub["Retorno Anualizado"].mean(),
        "Volatilidad anual prom.":  sub["Volatilidad Anual"].mean(),
        "Sharpe promedio":          sub["Sharpe Ratio"].mean(),
        "Max DD promedio":          sub["Max Drawdown"].mean(),
        "Correlación interna prom.": c_prom,
    }

tickers_arg_ok = [t for t in TICKERS_ARG if t in metricas_df.index]
tickers_int_ok = [t for t in TICKERS_INT if t in metricas_df.index]

stats_arg = stats_grupo(tickers_arg_ok)
stats_int = stats_grupo(tickers_int_ok)

df_comp = pd.DataFrame({
    "🇦🇷 Argentinos":      stats_arg,
    "🌎 Internacionales":  stats_int,
}).T

fmt_comp = {
    "Retorno anual prom.":      "{:.2%}",
    "Volatilidad anual prom.":  "{:.2%}",
    "Sharpe promedio":          "{:.3f}",
    "Max DD promedio":          "{:.2%}",
    "Correlación interna prom.": "{:.3f}",
}
print("Comparación de grupos:")
print(df_comp.style.format(fmt_comp).to_string())

In [ ]:
# ============================================================
# GRÁFICO: ARGENTINA VS EXTERIOR
# ============================================================

metricas_comp = ["Retorno anual prom.", "Volatilidad anual prom.",
                 "Sharpe promedio", "Max DD promedio"]

vals_arg = [stats_arg.get(m, 0) for m in metricas_comp]
vals_int = [stats_int.get(m, 0) for m in metricas_comp]

x = np.arange(len(metricas_comp))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, [v * 100 for v in vals_arg], width,
               label='🇦🇷 Argentinos', color='#FF6B35', alpha=0.85)
bars2 = ax.bar(x + width/2, [v * 100 for v in vals_int], width,
               label='🌎 Internacionales', color='#2196F3', alpha=0.85)

ax.bar_label(bars1, fmt='%.1f', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.1f', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(metricas_comp, rotation=15, ha='right', fontsize=10)
ax.set_ylabel("Valor (%)", fontsize=11)
ax.set_title("Comparación de Métricas Promedio: Argentina vs. Exterior",
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.axhline(0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.savefig("grafico_arg_vs_ext.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 12. Retornos Acumulados Finales

In [ ]:
# ============================================================
# RETORNOS ACUMULADOS POR ACTIVO
# ============================================================

ret_acum = (1 + log_returns).cumprod() - 1

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Argentinos
for i, t in enumerate([t for t in TICKERS_ARG if t in ret_acum.columns]):
    axes[0].plot(ret_acum.index, ret_acum[t] * 100, label=t,
                 color=colores_arg[i % len(colores_arg)], linewidth=1.8)
axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
axes[0].set_title("🇦🇷 Retornos Acumulados — Argentinos", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Retorno acumulado (%)")
axes[0].legend(fontsize=9)

# Internacionales
for i, t in enumerate([t for t in TICKERS_INT if t in ret_acum.columns]):
    axes[1].plot(ret_acum.index, ret_acum[t] * 100, label=t,
                 color=colores_int[i % len(colores_int)], linewidth=1.8)
axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
axes[1].set_title("🌎 Retornos Acumulados — Internacionales", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Retorno acumulado (%)")
axes[1].legend(fontsize=9)

plt.suptitle("Retornos Acumulados por Grupo", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("grafico_retornos_acumulados.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 13. Conclusiones

### 13.1 Hallazgos principales

Este notebook demostró la aplicación completa de la Teoría Moderna de Portafolios de Markowitz sobre un conjunto mixto de activos argentinos e internacionales:

1. **La diversificación reduce el riesgo**: el portafolio de máximo Sharpe logra un Sharpe Ratio superior al de cualquier activo individual, gracias a la combinación de activos con baja correlación entre sí.

2. **La optimización supera al benchmark equiponderado**: tanto el portafolio de máximo Sharpe como el de mínima varianza dominan al equiponderado en términos de Sharpe Ratio, confirmando que la distribución naïve 1/N es sub-óptima.

3. **Activos argentinos vs. internacionales**: los ADRs argentinos exhiben mayor volatilidad promedio, consistente con el riesgo país, pero algunos ofrecen retornos que compensan ese riesgo adicional cuando se combinan correctamente con activos internacionales de menor correlación.

4. **La Capital Market Line**: todo portafolio a lo largo de la CML es óptimo en el sentido de Markowitz: maximiza el Sharpe para su nivel de riesgo.

### 13.2 Librerías del curso utilizadas

| Librería | Uso en este proyecto |
|----------|---------------------|
| `pandas` | Manejo de series de precios y retornos |
| `numpy` | Álgebra matricial (covarianzas, pesos) |
| `matplotlib` / `seaborn` | Visualizaciones estáticas |
| `scipy.optimize` | Optimización SLSQP |
| `yfinance` | Descarga de datos reales |

### 13.3 Limitaciones del modelo

- **Normalidad**: los retornos reales presentan colas pesadas no capturadas por el modelo
- **Estacionariedad**: las correlaciones varían en el tiempo; el modelo las trata como constantes
- **Prospectivo vs. retrospectivo**: el portafolio óptimo histórico no necesariamente es óptimo en el futuro
- **Costos omitidos**: comisiones, spreads y taxes no están incluidos

### 13.4 Extensiones posibles

- Incorporar restricciones de turnover para carteras rebalanceables
- Aplicar Black-Litterman para incorporar visiones del inversor
- Usar modelos de volatilidad estocástica (GARCH) para matrices de covarianza dinámicas
- Agregar análisis de sensibilidad de los pesos ante cambios en las estimaciones

---

> ⚠️ **Advertencia final:** Todos los resultados son exclusivamente educativos. Este notebook forma parte del Challenge de la materia Economía Computacional (UADE, 2025) y no constituye asesoramiento financiero de ningún tipo. El desempeño pasado no garantiza resultados futuros.